# NSG-IDS: Neuro-Symbolic Generative Framework for Intrusion Detection
**Full implementation of the NSG-IDS paper.**
Datasets expected in `./dataset/` sub-directories:
- `./dataset/CIC-IDS-2018/` — any CSV with CICFlowMeter columns + `Label`
- `./dataset/UNSW-NB15/` — any CSV with UNSW-NB15 columns + `attack_cat`
- `./dataset/NF-ToN-IoT/` — any CSV with NetFlow columns + `Label`

If no files are found, realistic demo data is generated automatically.

In [1]:
import os, warnings, json, time, subprocess, sys



import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from pathlib import Path
from collections import defaultdict
from copy import deepcopy
from sklearn.preprocessing import LabelEncoder, MinMaxScaler
from sklearn.mixture import BayesianGaussianMixture
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score, precision_score, recall_score, accuracy_score
from sklearn.manifold import TSNE
from scipy.stats import ks_2samp
from scipy.spatial.distance import jensenshannon
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils import spectral_norm
from tqdm import tqdm
warnings.filterwarnings('ignore')

DEVICE = 'cuda' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu'
print(f'Device: {DEVICE}')
print(f'Torch: {torch.__version__} | CUDA: {torch.cuda.is_available()} | MPS: {torch.backends.mps.is_available()}')



# ── reproducibility ──
SEED = 42
np.random.seed(SEED); torch.manual_seed(SEED)

# ── output dirs ──
for d in ['./outputs', './models', './figures']:
    os.makedirs(d, exist_ok=True)
print('Directories ready.')

Device: mps
Torch: 2.8.0 | CUDA: False | MPS: True
Directories ready.


In [2]:
# ══════════════════════════════════════════════════════════════
#  CONFIGURATION  (edit here to tune)
# ══════════════════════════════════════════════════════════════
CFG = dict(
    # paths
    DATASET_DIR   = './dataset',
    # CIF-32 preprocessing
    K_VGM         = 5,    # Gaussian components per continuous feature
    N_BINS        = 32,   # equipopulation bins for discrete-int features
    # architecture
    Z_DIM         = 128,  # latent noise dim
    EMBED_DIM     = 64,   # class-source embedding dim
    HIDDEN_DIM    = 512,
    N_RES_BLOCKS  = 4,
    ATTN_HEADS    = 8,
    ATTN_FEAT_DIM = 32,   # d_e per CIF feature after reshape for MHA
    DROPOUT       = 0.1,
    # training
    BATCH_SIZE    = 512,
    N_EPOCHS      = 300,
    CRITIC_STEPS  = 5,
    LAMBDA_GP     = 10.0,
    LR            = 1e-4,
    BETAS         = (0.5, 0.9),
    TAU_START     = 1.0,
    TAU_END       = 0.2,
    # evaluation
    N_SYNTH_PER_CLASS = 20_000,
    N_RUNS            = 3,
    COVERAGE_MIN      = 500,
    RF_TREES          = 500,
    # demo / fast-test
    DEMO_ROWS_PER_SRC = 8_000,   # rows per source when no real data found
    FAST_EPOCHS       = 10,      # set > 0 to override N_EPOCHS for quick test; 0 = full N_EPOCHS
    # ── data fraction: change DATA_PCT to 0.10 | 0.20 | 0.50 | 1.00
    DATA_PCT          = 1.00,    # fraction of training data to use
)
print('Config loaded.')

Config loaded.


## 1 · Data Loading & CIF-32 Harmonisation

In [3]:
# ── CIF-32 column definitions ──────────────────────────────────
# Each entry: (cif_name, type)  where type ∈ {'cat','cont','int','bin'}
CIF32 = [
    ('protocol',         'cat'),   # 0
    ('src_port_cat',     'cat'),   # 1
    ('dst_port_cat',     'cat'),   # 2
    ('flow_duration',    'cont'),  # 3
    ('iat_mean',         'cont'),  # 4
    ('iat_std',          'cont'),  # 5
    ('fwd_pkts',         'int'),   # 6
    ('fwd_bytes',        'int'),   # 7
    ('fwd_pkt_len_mean', 'cont'),  # 8
    ('fwd_pkt_len_std',  'cont'),  # 9
    ('bwd_pkts',         'int'),   # 10
    ('bwd_bytes',        'int'),   # 11
    ('bwd_pkt_len_mean', 'cont'),  # 12
    ('bwd_pkt_len_std',  'cont'),  # 13
    ('bytes_per_sec',    'cont'),  # 14
    ('pkts_per_sec',     'cont'),  # 15
    ('fwd_bwd_ratio',   'cont'),   # 16
    ('down_up_ratio',   'cont'),   # 17
    ('syn_flag',         'bin'),   # 18
    ('fin_flag',         'bin'),   # 19
    ('rst_flag',         'bin'),   # 20
    ('psh_flag',         'bin'),   # 21
    ('ack_flag',         'bin'),   # 22
    ('urg_flag',         'bin'),   # 23
    ('subflow_fwd_pkts', 'int'),   # 24
    ('subflow_bwd_pkts', 'int'),   # 25
    ('init_win_fwd',     'int'),   # 26
    ('init_win_bwd',     'int'),   # 27
    ('ttl_fwd',          'cont'),  # 28
    ('ttl_bwd',          'cont'),  # 29
    ('fwd_bulk_rate',    'cont'),  # 30
    ('bwd_bulk_rate',    'cont'),  # 31
]
CIF_NAMES  = [c for c,_ in CIF32]
CIF_TYPES  = {c:t for c,t in CIF32}
CONT_COLS  = [c for c,t in CIF32 if t=='cont']
INT_COLS   = [c for c,t in CIF32 if t=='int']
CAT_COLS   = [c for c,t in CIF32 if t=='cat']
BIN_COLS   = [c for c,t in CIF32 if t=='bin']
print(f'CIF-32: {len(CONT_COLS)} cont, {len(INT_COLS)} int, {len(CAT_COLS)} cat, {len(BIN_COLS)} bin')

# Protocol map (derived from port / proto number)
def proto_cat(p):
    p = str(p).strip().upper()
    if p in ('6','TCP'):   return 0
    if p in ('17','UDP'):  return 1
    return 2  # ICMP / other

def port_cat(p):
    try: p = int(float(p))
    except: return 2
    if p < 1024:   return 0  # well-known
    if p < 49152:  return 1  # registered
    return 2                 # ephemeral

CIF-32: 15 cont, 8 int, 3 cat, 6 bin


In [4]:
# ── per-source CIF-32 mapping functions ────────────────────────

def map_cic2018(df):
    """Map CIC-IDS-2018 CICFlowMeter columns → CIF-32.
    Handles both full names ('Total Fwd Packets') and
    abbreviated names ('Tot Fwd Pkts', 'Flow Byts/s', 'Init Fwd Win Byts', etc.)."""
    df.columns = df.columns.str.strip()
    label_col = next((c for c in df.columns if c.strip().lower() == 'label'), df.columns[-1])
    df = df.replace([float('inf'), float('-inf')], float('nan')).dropna(subset=[label_col])

    ALIASES = {
        'Protocol':                    ['Protocol'],
        'Source Port':                 ['Source Port', 'Src Port', 'SrcPort'],
        'Destination Port':            ['Destination Port', 'Dst Port', 'DestPort'],
        'Flow Duration':               ['Flow Duration'],
        'Flow IAT Mean':               ['Flow IAT Mean'],
        'Flow IAT Std':                ['Flow IAT Std'],
        'Total Fwd Packets':           ['Total Fwd Packets', 'Tot Fwd Pkts'],
        'Total Length of Fwd Packets': ['Total Length of Fwd Packets', 'TotLen Fwd Pkts'],
        'Fwd Packet Length Mean':      ['Fwd Packet Length Mean', 'Fwd Pkt Len Mean'],
        'Fwd Packet Length Std':       ['Fwd Packet Length Std', 'Fwd Pkt Len Std'],
        'Total Backward Packets':      ['Total Backward Packets', 'Tot Bwd Pkts'],
        'Total Length of Bwd Packets': ['Total Length of Bwd Packets', 'TotLen Bwd Pkts'],
        'Bwd Packet Length Mean':      ['Bwd Packet Length Mean', 'Bwd Pkt Len Mean'],
        'Bwd Packet Length Std':       ['Bwd Packet Length Std', 'Bwd Pkt Len Std'],
        'Flow Bytes/s':                ['Flow Bytes/s', 'Flow Byts/s'],
        'Flow Packets/s':              ['Flow Packets/s', 'Flow Pkts/s'],
        'Down/Up Ratio':               ['Down/Up Ratio'],
        'SYN Flag Count':              ['SYN Flag Count', 'SYN Flag Cnt'],
        'FIN Flag Count':              ['FIN Flag Count', 'FIN Flag Cnt'],
        'RST Flag Count':              ['RST Flag Count', 'RST Flag Cnt'],
        'PSH Flag Count':              ['PSH Flag Count', 'PSH Flag Cnt'],
        'ACK Flag Count':              ['ACK Flag Count', 'ACK Flag Cnt'],
        'URG Flag Count':              ['URG Flag Count', 'URG Flag Cnt'],
        'Subflow Fwd Packets':         ['Subflow Fwd Packets', 'Subflow Fwd Pkts'],
        'Subflow Bwd Packets':         ['Subflow Bwd Packets', 'Subflow Bwd Pkts'],
        'Init_Win_bytes_forward':      ['Init_Win_bytes_forward', 'Init Fwd Win Byts'],
        'Init_Win_bytes_backward':     ['Init_Win_bytes_backward', 'Init Bwd Win Byts'],
        'Fwd Avg Bulk Rate':           ['Fwd Avg Bulk Rate', 'Fwd Blk Rate Avg'],
        'Bwd Avg Bulk Rate':           ['Bwd Avg Bulk Rate', 'Bwd Blk Rate Avg'],
    }
    col_map = {c.strip().lower().replace(' ', '').replace('_', ''): c for c in df.columns}

    def gc(canonical, default=0.0):
        for alias in ALIASES.get(canonical, [canonical]):
            key = alias.lower().replace(' ', '').replace('_', '')
            if key in col_map:
                return df[col_map[key]].fillna(default).astype(float)
        return pd.Series(default, index=df.index)

    out = pd.DataFrame()
    out['protocol']         = gc('Protocol', 6).apply(proto_cat)
    out['src_port_cat']     = gc('Source Port', 0).apply(port_cat)
    out['dst_port_cat']     = gc('Destination Port', 0).apply(port_cat)
    dur                     = gc('Flow Duration', 1).clip(lower=1e-6)
    out['flow_duration']    = dur
    out['iat_mean']         = gc('Flow IAT Mean', 0)
    out['iat_std']          = gc('Flow IAT Std', 0)
    out['fwd_pkts']         = gc('Total Fwd Packets', 0)
    out['fwd_bytes']        = gc('Total Length of Fwd Packets', 0)
    out['fwd_pkt_len_mean'] = gc('Fwd Packet Length Mean', 0)
    out['fwd_pkt_len_std']  = gc('Fwd Packet Length Std', 0)
    out['bwd_pkts']         = gc('Total Backward Packets', 0)
    out['bwd_bytes']        = gc('Total Length of Bwd Packets', 0)
    out['bwd_pkt_len_mean'] = gc('Bwd Packet Length Mean', 0)
    out['bwd_pkt_len_std']  = gc('Bwd Packet Length Std', 0)
    out['bytes_per_sec']    = gc('Flow Bytes/s', 0).replace([float('inf'), float('-inf')], 0)
    out['pkts_per_sec']     = gc('Flow Packets/s', 0).replace([float('inf'), float('-inf')], 0)
    fwd = out['fwd_pkts'].clip(lower=0)
    bwd = out['bwd_pkts'].clip(lower=0)
    out['fwd_bwd_ratio']    = fwd / (bwd + 1)
    out['down_up_ratio']    = gc('Down/Up Ratio', 1)
    out['syn_flag']         = (gc('SYN Flag Count', 0) > 0).astype(int)
    out['fin_flag']         = (gc('FIN Flag Count', 0) > 0).astype(int)
    out['rst_flag']         = (gc('RST Flag Count', 0) > 0).astype(int)
    out['psh_flag']         = (gc('PSH Flag Count', 0) > 0).astype(int)
    out['ack_flag']         = (gc('ACK Flag Count', 0) > 0).astype(int)
    out['urg_flag']         = (gc('URG Flag Count', 0) > 0).astype(int)
    out['subflow_fwd_pkts'] = gc('Subflow Fwd Packets', 0)
    out['subflow_bwd_pkts'] = gc('Subflow Bwd Packets', 0)
    out['init_win_fwd']     = gc('Init_Win_bytes_forward', 0).clip(lower=0)
    out['init_win_bwd']     = gc('Init_Win_bytes_backward', 0).clip(lower=0)
    out['ttl_fwd']          = 64.0
    out['ttl_bwd']          = 64.0
    out['fwd_bulk_rate']    = gc('Fwd Avg Bulk Rate', 0)
    out['bwd_bulk_rate']    = gc('Bwd Avg Bulk Rate', 0)
    out['label']            = df[label_col].fillna('BENIGN').astype(str).str.strip()
    out['source']           = 0
    out['missingness']      = 0
    return out.reset_index(drop=True)


def map_unsw(df):
    """Map UNSW-NB15 columns → CIF-32."""
    df.columns = df.columns.str.strip().str.lower()
    df = df.replace([float('inf'), float('-inf')], float('nan'))
    def gc(name, default=0.0):
        return df[name].fillna(default).astype(float) if name in df.columns else pd.Series(default, index=df.index)
    def gcs(name, default=''):
        return df[name].fillna(default).astype(str) if name in df.columns else pd.Series(default, index=df.index)
    out = pd.DataFrame()
    out['protocol']         = gcs('proto', 'tcp').apply(proto_cat)
    out['src_port_cat']     = gc('sport', 0).apply(port_cat)
    out['dst_port_cat']     = gc('dsport', 0).apply(port_cat)
    out['flow_duration']    = gc('dur', 1).clip(lower=1e-6)
    out['iat_mean']         = gc('sinpkt', 0)
    out['iat_std']          = gc('dinpkt', 0)
    out['fwd_pkts']         = gc('spkts', 0)
    out['fwd_bytes']        = gc('sbytes', 0)
    out['fwd_pkt_len_mean'] = gc('smean', 0)
    out['fwd_pkt_len_std']  = gc('sjit', 0)
    out['bwd_pkts']         = gc('dpkts', 0)
    out['bwd_bytes']        = gc('dbytes', 0)
    out['bwd_pkt_len_mean'] = gc('dmean', 0)
    out['bwd_pkt_len_std']  = gc('djit', 0)
    dur = out['flow_duration'].clip(lower=1e-6)
    out['bytes_per_sec']    = (out['fwd_bytes'] + out['bwd_bytes']) / dur
    out['pkts_per_sec']     = (out['fwd_pkts'] + out['bwd_pkts']) / dur
    out['fwd_bwd_ratio']    = out['fwd_pkts'] / (out['bwd_pkts'] + 1)
    out['down_up_ratio']    = gc('dload', 0) / (gc('sload', 1) + 1e-6)
    state = gcs('state', 'CON')
    out['syn_flag']  = state.str.contains('SYN|INT', case=False).astype(int)
    out['fin_flag']  = state.str.contains('FIN|CLO', case=False).astype(int)
    out['rst_flag']  = state.str.contains('RST',     case=False).astype(int)
    out['psh_flag']  = 0
    out['ack_flag']  = state.str.contains('CON|ECO', case=False).astype(int)
    out['urg_flag']  = 0
    out['subflow_fwd_pkts'] = gc('spkts', 0) // 2
    out['subflow_bwd_pkts'] = gc('dpkts', 0) // 2
    out['init_win_fwd']     = gc('swin', 0)
    out['init_win_bwd']     = gc('dwin', 0)
    out['ttl_fwd']          = gc('sttl', 64)
    out['ttl_bwd']          = gc('dttl', 64)
    out['fwd_bulk_rate']    = gc('sload', 0)
    out['bwd_bulk_rate']    = gc('dload', 0)
    label_col = 'attack_cat' if 'attack_cat' in df.columns else 'label'
    out['label']  = df[label_col].fillna('Normal').astype(str).str.strip().replace({'': 'Normal', 'nan': 'Normal'})
    out['source'] = 1
    out['missingness'] = 0
    return out.reset_index(drop=True)


def map_toniot(df):
    """Map NF-ToN-IoT NetFlow columns → CIF-32."""
    df.columns = df.columns.str.strip()
    df = df.replace([float('inf'), float('-inf')], float('nan'))
    def gc(name, default=0.0):
        matches = [c for c in df.columns if c.strip().upper() == name.upper()]
        return df[matches[0]].fillna(default).astype(float) if matches else pd.Series(default, index=df.index)
    out = pd.DataFrame()
    out['protocol']         = gc('PROTOCOL', 6).apply(proto_cat)
    out['src_port_cat']     = gc('L4_SRC_PORT', 0).apply(port_cat)
    out['dst_port_cat']     = gc('L4_DST_PORT', 0).apply(port_cat)
    dur_s                   = gc('FLOW_DURATION_MILLISECONDS', 1).clip(lower=1) / 1000.0
    out['flow_duration']    = dur_s
    out['iat_mean']         = gc('DURATION_IN', 0)
    out['iat_std']          = gc('DURATION_OUT', 0)
    out['fwd_pkts']         = gc('IN_PKTS', 0)
    out['fwd_bytes']        = gc('IN_BYTES', 0)
    out['fwd_pkt_len_mean'] = out['fwd_bytes'] / (out['fwd_pkts'] + 1)
    out['fwd_pkt_len_std']  = 0.0
    out['bwd_pkts']         = gc('OUT_PKTS', 0)
    out['bwd_bytes']        = gc('OUT_BYTES', 0)
    out['bwd_pkt_len_mean'] = out['bwd_bytes'] / (out['bwd_pkts'] + 1)
    out['bwd_pkt_len_std']  = 0.0
    out['bytes_per_sec']    = (out['fwd_bytes'] + out['bwd_bytes']) / dur_s.clip(lower=1e-6)
    out['pkts_per_sec']     = (out['fwd_pkts'] + out['bwd_pkts']) / dur_s.clip(lower=1e-6)
    out['fwd_bwd_ratio']    = out['fwd_pkts'] / (out['bwd_pkts'] + 1)
    out['down_up_ratio']    = gc('DST_TO_SRC_AVG_THROUGHPUT', 0) / (gc('SRC_TO_DST_AVG_THROUGHPUT', 1) + 1e-6)
    flags = gc('TCP_FLAGS', 0).astype(int)
    out['syn_flag']         = ((flags & 0x02) > 0).astype(int)
    out['fin_flag']         = ((flags & 0x01) > 0).astype(int)
    out['rst_flag']         = ((flags & 0x04) > 0).astype(int)
    out['psh_flag']         = ((flags & 0x08) > 0).astype(int)
    out['ack_flag']         = ((flags & 0x10) > 0).astype(int)
    out['urg_flag']         = ((flags & 0x20) > 0).astype(int)
    out['subflow_fwd_pkts'] = out['fwd_pkts'] // 2
    out['subflow_bwd_pkts'] = out['bwd_pkts'] // 2
    out['init_win_fwd']     = gc('TCP_WIN_MAX_IN', 0)
    out['init_win_bwd']     = gc('TCP_WIN_MAX_OUT', 0)
    out['ttl_fwd']          = gc('MIN_TTL', 64)
    out['ttl_bwd']          = gc('MAX_TTL', 64)
    out['fwd_bulk_rate']    = gc('SRC_TO_DST_SECOND_BYTES', 0)
    out['bwd_bulk_rate']    = gc('DST_TO_SRC_SECOND_BYTES', 0)
    label_col = next((c for c in df.columns if c.strip().lower() == 'label'),
                     next((c for c in df.columns if c.strip().lower() == 'attack'), df.columns[-1]))
    out['label']       = df[label_col].fillna('Benign').astype(str).str.strip()
    out['source']      = 2
    out['missingness'] = 1
    return out.reset_index(drop=True)


In [5]:
# ── demo data generator (used when real CSVs not found) ────────
def make_demo_data(n=CFG['DEMO_ROWS_PER_SRC'], seed=42):
    rng = np.random.default_rng(seed)
    src_info = [
        (0, ['BENIGN','DoS Hulk','PortScan','DDoS','FTP-Patator','SSH-Patator',
              'Bot','Infiltration','Heartbleed','XSS','SQL Injection',
              'Brute Force','DoS Slowloris','Web Attack']),
        (1, ['Normal','Fuzzers','Analysis','Backdoors','DoS','Exploits',
              'Generic','Reconnaissance','Shellcode','Worms']),
        (2, ['Benign','DoS','DDoS','Reconnaissance','Backdoor','Injection',
              'MITM','Ransomware','Password','XSS','Scanning']),
    ]
    frames = []
    for src_id, labels in src_info:
        # imbalanced: benign >> attacks
        weights = np.array([10.0] + [1.0]*(len(labels)-1))
        weights /= weights.sum()
        chosen = rng.choice(labels, size=n, p=weights)
        row = {}
        row['protocol']         = rng.integers(0,3,n)
        row['src_port_cat']     = rng.integers(0,3,n)
        row['dst_port_cat']     = rng.integers(0,3,n)
        row['flow_duration']    = rng.exponential(1e5, n)
        row['iat_mean']         = rng.exponential(5e4, n)
        row['iat_std']          = rng.exponential(3e4, n)
        row['fwd_pkts']         = rng.integers(1,500,n)
        row['fwd_bytes']        = rng.integers(0,500000,n)
        row['fwd_pkt_len_mean'] = rng.uniform(0,1500,n)
        row['fwd_pkt_len_std']  = rng.uniform(0,500,n)
        row['bwd_pkts']         = rng.integers(0,400,n)
        row['bwd_bytes']        = rng.integers(0,400000,n)
        row['bwd_pkt_len_mean'] = rng.uniform(0,1500,n)
        row['bwd_pkt_len_std']  = rng.uniform(0,500,n)
        row['bytes_per_sec']    = rng.exponential(1e4, n)
        row['pkts_per_sec']     = rng.exponential(100, n)
        row['fwd_bwd_ratio']    = rng.exponential(2, n)
        row['down_up_ratio']    = rng.exponential(1, n)
        for f in ['syn_flag','fin_flag','rst_flag','psh_flag','ack_flag','urg_flag']:
            row[f] = rng.integers(0,2,n)
        row['subflow_fwd_pkts'] = rng.integers(0,100,n)
        row['subflow_bwd_pkts'] = rng.integers(0,100,n)
        row['init_win_fwd']     = rng.integers(0,65536,n)
        row['init_win_bwd']     = rng.integers(0,65536,n)
        row['ttl_fwd']          = rng.choice([64,128,255], n).astype(float)
        row['ttl_bwd']          = rng.choice([64,128,255], n).astype(float)
        row['fwd_bulk_rate']    = rng.exponential(5000, n)
        row['bwd_bulk_rate']    = rng.exponential(5000, n)
        row['label']            = chosen
        row['source']           = src_id
        row['missingness']      = int(src_id == 2)
        frames.append(pd.DataFrame(row))
    return pd.concat(frames, ignore_index=True)

In [6]:
# ── main loader ────────────────────────────────────────────────
def load_source(subdir, map_fn, max_rows=None):
    p = Path(CFG['DATASET_DIR']) / subdir
    csvs = list(p.glob('*.csv')) + list(p.glob('*.CSV')) if p.exists() else []
    if not csvs:
        return None
    dfs = []
    for f in csvs:
        try:
            chunk = pd.read_csv(f, low_memory=False)
            if max_rows: chunk = chunk.sample(min(max_rows, len(chunk)), random_state=SEED)
            dfs.append(map_fn(chunk))
            print(f'  Loaded {f.name}: {len(chunk):,} rows')
        except Exception as e:
            print(f'  Warning: skipped {f.name}: {e}')
    return pd.concat(dfs, ignore_index=True) if dfs else None

print('Loading datasets...')
MAX_ROWS = 500_000  # per CSV; set None for full dataset
cic   = load_source('CIC-IDS-2018',  map_cic2018, MAX_ROWS)
unsw  = load_source('UNSW-NB15',     map_unsw,    MAX_ROWS)
toniot= load_source('NF-ToN-IoT',    map_toniot,  MAX_ROWS)

if cic is None and unsw is None and toniot is None:
    print('No datasets found → generating demo data (DEMO_MODE=True)')
    CFG['DEMO_MODE'] = True
    fused = make_demo_data()
else:
    parts = [df for df in [cic, unsw, toniot] if df is not None]
    fused = pd.concat(parts, ignore_index=True)
    print(f'Fused corpus: {len(fused):,} rows')

# clip extreme values
for c in CONT_COLS + INT_COLS:
    fused[c] = pd.to_numeric(fused[c], errors='coerce').fillna(0).clip(lower=0)
fused['label']  = fused['label'].astype(str).str.strip()
fused['source'] = fused['source'].astype(int)

# encode class labels
label_enc = LabelEncoder()
fused['class_id'] = label_enc.fit_transform(fused['label'])
source_enc = LabelEncoder()
fused['source_id'] = source_enc.fit_transform(fused['source'])

N_CLASSES = len(label_enc.classes_)
N_SOURCES = len(source_enc.classes_)
print(f'Classes: {N_CLASSES}  |  Sources: {N_SOURCES}')
print(label_enc.classes_)

# train / test split (stratified 80/20)
from sklearn.model_selection import train_test_split
train_df, test_df = train_test_split(fused, test_size=0.2, stratify=fused['class_id'], random_state=SEED)
print(f'Train: {len(train_df):,}  |  Test: {len(test_df):,}')

# ── DATA_PCT subsampling ────────────────────────────────────────────────────
_pct = CFG.get('DATA_PCT', 1.0)
if _pct < 1.0:
    from sklearn.model_selection import train_test_split as _tts
    _, train_df = _tts(train_df, test_size=_pct,
                       stratify=train_df['class_id'], random_state=SEED)
    train_df = train_df.reset_index(drop=True)
    print(f'DATA_PCT={_pct*100:.0f}%  ->  training rows: {len(train_df):,}')
else:
    print(f'DATA_PCT=100%  ->  training rows: {len(train_df):,}')

# per-source test subsets
test_by_src = {s: test_df[test_df['source']==s] for s in test_df['source'].unique()}

Loading datasets...
  Loaded cic.csv: 500,000 rows
  Loaded UNSW_NB15_training-set.csv: 175,341 rows
Fused corpus: 675,341 rows
Classes: 13  |  Sources: 2
['Analysis' 'Backdoor' 'Benign' 'DoS' 'Exploits' 'FTP-BruteForce'
 'Fuzzers' 'Generic' 'Normal' 'Reconnaissance' 'SSH-Bruteforce'
 'Shellcode' 'Worms']
Train: 540,272  |  Test: 135,069
DATA_PCT=100%  ->  training rows: 540,272


## 2 · Preprocessing — VGM, Binning, One-Hot

In [7]:
# ── Variational Gaussian Mixture transformer ───────────────────
class VGMTransformer:
    """VGM normalization per continuous column (CTGAN-style)."""
    def __init__(self, K=5, max_iter=100):
        self.K = K; self.bgm = {}

    def fit(self, df, cols):
        for c in cols:
            v = df[c].values.reshape(-1,1)
            bgm = BayesianGaussianMixture(n_components=self.K, n_init=1,
                                          max_iter=100, random_state=SEED)
            bgm.fit(v)
            self.bgm[c] = bgm
        return self

    def transform(self, df, cols):
        parts = []
        self.col_slices = {}
        offset = 0
        for c in cols:
            bgm = self.bgm[c]
            v   = df[c].values.reshape(-1,1)
            probs = bgm.predict_proba(v)          # (N, K)
            mode  = np.argmax(probs, axis=1)       # (N,)
            means = bgm.means_.flatten()
            stds  = np.sqrt(bgm.covariances_.flatten())
            norm  = (v.flatten() - means[mode]) / (4 * stds[mode] + 1e-6)
            norm  = np.clip(norm, -1, 1)
            onehot = np.zeros((len(v), self.K), dtype=np.float32)
            onehot[np.arange(len(v)), mode] = 1.0
            block = np.column_stack([onehot, norm.astype(np.float32)])  # (N, K+1)
            parts.append(block)
            self.col_slices[c] = (offset, offset + self.K + 1)
            offset += self.K + 1
        return np.column_stack(parts) if parts else np.zeros((len(df), 0))

    def inverse(self, arr, cols):
        out = {}
        for c in cols:
            s, e = self.col_slices[c]
            bgm  = self.bgm[c]
            oh   = arr[:, s:s+self.K]
            norm = arr[:, s+self.K]
            mode = np.argmax(oh, axis=1)
            means = bgm.means_.flatten()
            stds  = np.sqrt(bgm.covariances_.flatten())
            out[c] = norm * 4 * stds[mode] + means[mode]
        return out


# ── Equipopulation bin transformer ────────────────────────────
class BinTransformer:
    """Bin discrete-integer columns into B equipopulation bins."""
    def __init__(self, B=32):
        self.B = B; self.edges = {}

    def fit(self, df, cols):
        for c in cols:
            v = df[c].values.astype(float)
            qs = np.linspace(0, 100, self.B+1)
            self.edges[c] = np.unique(np.percentile(v, qs))
        return self

    def transform(self, df, cols):
        parts = []
        self.col_slices = {}
        offset = 0
        for c in cols:
            v    = df[c].values.astype(float)
            bins = np.digitize(v, self.edges[c][1:-1])  # 0..B-1
            bins = np.clip(bins, 0, self.B-1)
            oh   = np.zeros((len(v), self.B), dtype=np.float32)
            oh[np.arange(len(v)), bins] = 1.0
            parts.append(oh)
            self.col_slices[c] = (offset, offset+self.B)
            offset += self.B
        return np.column_stack(parts) if parts else np.zeros((len(df),0))

    def inverse(self, arr, cols):
        out = {}
        for c in cols:
            s, e = self.col_slices[c]
            bins  = np.argmax(arr[:, s:e], axis=1)
            edges = self.edges[c]
            centroids = [(edges[i]+edges[i+1])/2 if i+1<len(edges) else edges[-1]
                         for i in range(min(self.B, len(edges)-1))]
            out[c] = np.array([centroids[min(b, len(centroids)-1)] for b in bins])
        return out


# ── CIF-32 Preprocessor (combines all transformers) ───────────
class CIF32Preprocessor:
    def __init__(self, K=5, B=32):
        self.vgm = VGMTransformer(K=K)
        self.bins = BinTransformer(B=B)
        self.cat_encoders = {c: LabelEncoder() for c in CAT_COLS}
        self.n_cat_classes = {}
        self.fitted = False
        self.output_info = []   # list of (dim, type_str) for generator output heads

    def fit(self, df):
        print('Fitting VGM transformers...')
        self.vgm.fit(df, CONT_COLS)
        print('Fitting bin transformers...')
        self.bins.fit(df, INT_COLS)
        for c in CAT_COLS:
            self.cat_encoders[c].fit(df[c].astype(int).astype(str))
            self.n_cat_classes[c] = len(self.cat_encoders[c].classes_)
        self.fitted = True
        # build output_info in same column order as transform()
        self.output_info = []
        for c in CAT_COLS:
            self.output_info.append((self.n_cat_classes[c], 'softmax'))
        for c in CONT_COLS:
            self.output_info.append((5, 'softmax'))  # mode onehot
            self.output_info.append((1, 'tanh'))     # normalized value
        for c in INT_COLS:
            self.output_info.append((32, 'softmax'))
        for c in BIN_COLS:
            self.output_info.append((2, 'softmax'))  # binary → 2-class
        self.output_info.append((2, 'softmax'))      # missingness
        self.out_dim = sum(d for d,_ in self.output_info)
        print(f'Preprocessor fitted. Output dim = {self.out_dim}')
        return self

    def transform(self, df):
        parts = []
        # categorical (one-hot)
        for c in CAT_COLS:
            v = df[c].astype(int).astype(str)
            enc = self.cat_encoders[c].transform(v.map(lambda x: x if x in self.cat_encoders[c].classes_ else self.cat_encoders[c].classes_[0]))
            nc = self.n_cat_classes[c]
            oh = np.zeros((len(df), nc), dtype=np.float32)
            oh[np.arange(len(df)), enc] = 1.0
            parts.append(oh)
        # continuous (VGM)
        parts.append(self.vgm.transform(df, CONT_COLS).astype(np.float32))
        # discrete-int (bins)
        parts.append(self.bins.transform(df, INT_COLS).astype(np.float32))
        # binary (2-class one-hot)
        for c in BIN_COLS:
            v = df[c].astype(int).clip(0,1).values
            oh = np.zeros((len(df), 2), dtype=np.float32)
            oh[np.arange(len(df)), v] = 1.0
            parts.append(oh)
        # missingness
        v = df['missingness'].astype(int).clip(0,1).values
        oh = np.zeros((len(df), 2), dtype=np.float32)
        oh[np.arange(len(df)), v] = 1.0
        parts.append(oh)
        return np.concatenate(parts, axis=1)

print('Preprocessor classes defined.')

Preprocessor classes defined.


In [8]:
# ── clean data before preprocessing ──────────────────────────
# Replace inf values with the 99th percentile per column
for c in CONT_COLS:
    col_data = train_df[c].copy()
    # Replace inf with NaN first
    col_data = col_data.replace([np.inf, -np.inf], np.nan)
    # Get 99th percentile of finite values
    percentile_99 = col_data[np.isfinite(col_data)].quantile(0.99)
    # Fill NaN with median of finite values
    median = col_data[np.isfinite(col_data)].median()
    train_df[c] = col_data.fillna(median).clip(lower=0, upper=percentile_99)

print('Data cleaned: inf and nan values handled.')

# ── fit preprocessor on training data ─────────────────────────
prep = CIF32Preprocessor(K=CFG['K_VGM'], B=CFG['N_BINS'])
prep.fit(train_df)

print('Transforming train/test sets...')
X_train = prep.transform(train_df)
X_test  = prep.transform(test_df)
y_train = train_df['class_id'].values
y_test  = test_df['class_id'].values
src_train = train_df['source_id'].values
print(f'X_train shape: {X_train.shape}  |  X_test shape: {X_test.shape}')

Data cleaned: inf and nan values handled.
Fitting VGM transformers...
Fitting bin transformers...
Preprocessor fitted. Output dim = 367
Transforming train/test sets...
X_train shape: (540272, 367)  |  X_test shape: (135069, 367)


## 3 · NSG-IDS Model Architecture

In [9]:
# ── building blocks ────────────────────────────────────────────
class ResBlock(nn.Module):
    def __init__(self, dim, use_bn=True, dropout=0.1):
        super().__init__()
        layers = [nn.Linear(dim, dim)]
        if use_bn: layers.append(nn.BatchNorm1d(dim))
        layers += [nn.LeakyReLU(0.2), nn.Dropout(dropout), nn.Linear(dim, dim)]
        if use_bn: layers.append(nn.BatchNorm1d(dim))
        layers.append(nn.LeakyReLU(0.2))
        self.net = nn.Sequential(*layers)

    def forward(self, x): return x + self.net(x)


class SelfAttnModule(nn.Module):
    """Reshape → MHA over 32 CIF feature slots → reshape back."""
    def __init__(self, n_feat=32, feat_dim=32, n_heads=8):
        super().__init__()
        self.n_feat  = n_feat
        self.d       = feat_dim
        self.proj_in = nn.Linear(n_feat * feat_dim, n_feat * feat_dim)  # id; keeps dim
        self.attn    = nn.MultiheadAttention(feat_dim, n_heads, batch_first=True)
        self.norm1   = nn.LayerNorm(feat_dim)
        self.ffn     = nn.Sequential(nn.Linear(feat_dim, feat_dim*2), nn.GELU(),
                                     nn.Linear(feat_dim*2, feat_dim))
        self.norm2   = nn.LayerNorm(feat_dim)

    def forward(self, h):  # h: (B, n_feat*d)
        B = h.shape[0]
        H = h.view(B, self.n_feat, self.d)     # (B, 32, d)
        a, _ = self.attn(H, H, H)
        H    = self.norm1(H + a)
        H    = self.norm2(H + self.ffn(H))
        return H.reshape(B, -1)                # (B, 32*d)


class DiscreteAwareHead(nn.Module):
    """Per-CIF-feature output heads applying type-specific activation."""
    def __init__(self, in_dim, output_info):
        super().__init__()
        self.info  = output_info
        self.heads = nn.ModuleList([nn.Linear(in_dim, sz) for sz, _ in output_info])

    def forward(self, h, tau=1.0):
        outs = []
        for head, (sz, typ) in zip(self.heads, self.info):
            logits = head(h)
            if typ == 'softmax':
                outs.append(F.gumbel_softmax(logits, tau=tau, hard=False))
            else:  # tanh
                outs.append(torch.tanh(logits))
        return torch.cat(outs, dim=-1)


# ── Generator ─────────────────────────────────────────────────
class Generator(nn.Module):
    def __init__(self, z_dim, embed_dim, hidden_dim, n_blocks, output_info,
                 n_classes, n_sources, n_feat=32, feat_dim=32, n_heads=8,
                 dropout=0.1, use_attn=True, use_da=True):
        super().__init__()
        self.use_attn = use_attn
        self.use_da   = use_da
        self.n_feat   = n_feat
        self.feat_dim = feat_dim
        inp_dim = z_dim + embed_dim
        out_dim = sum(sz for sz,_ in output_info)

        # class-source embedding
        self.class_emb  = nn.Embedding(n_classes, embed_dim // 2)
        self.source_emb = nn.Embedding(n_sources, embed_dim // 2)

        # projection + residual
        self.proj   = nn.Linear(inp_dim, hidden_dim)
        self.blocks = nn.ModuleList([ResBlock(hidden_dim, use_bn=True, dropout=dropout)
                                     for _ in range(n_blocks)])
        # attention bridge: hidden_dim → n_feat*feat_dim
        attn_dim = n_feat * feat_dim
        self.to_attn   = nn.Linear(hidden_dim, attn_dim)
        if use_attn:
            self.attn_mod  = SelfAttnModule(n_feat, feat_dim, n_heads)
        self.from_attn = nn.Linear(attn_dim, hidden_dim)

        # output
        if use_da:
            self.out_head = DiscreteAwareHead(hidden_dim, output_info)
        else:
            self.out_head = nn.Sequential(nn.Linear(hidden_dim, out_dim), nn.Tanh())

    def forward(self, z, class_ids, source_ids, tau=1.0):
        e = torch.cat([self.class_emb(class_ids), self.source_emb(source_ids)], dim=-1)
        h = F.leaky_relu(self.proj(torch.cat([z, e], dim=-1)), 0.2)
        for blk in self.blocks: h = blk(h)
        a = self.to_attn(h)
        if self.use_attn: a = self.attn_mod(a)
        h = h + F.leaky_relu(self.from_attn(a), 0.2)
        if self.use_da:
            return self.out_head(h, tau)
        return self.out_head(h)


# ── Critic (Wasserstein) ───────────────────────────────────────
class Critic(nn.Module):
    def __init__(self, in_dim, embed_dim, hidden_dim, n_blocks,
                 n_classes, n_sources, n_feat=32, feat_dim=32,
                 n_heads=8, dropout=0.1):
        super().__init__()
        self.class_emb  = nn.Embedding(n_classes, embed_dim // 2)
        self.source_emb = nn.Embedding(n_sources, embed_dim // 2)
        inp = in_dim + embed_dim
        self.proj  = spectral_norm(nn.Linear(inp, hidden_dim))
        self.blocks = nn.ModuleList([
            nn.Sequential(
                spectral_norm(nn.Linear(hidden_dim, hidden_dim)),
                nn.LeakyReLU(0.2), nn.Dropout(dropout),
                spectral_norm(nn.Linear(hidden_dim, hidden_dim)),
                nn.LeakyReLU(0.2)
            ) for _ in range(n_blocks)
        ])
        attn_dim = n_feat * feat_dim
        self.to_attn   = spectral_norm(nn.Linear(hidden_dim, attn_dim))
        self.attn_mod  = SelfAttnModule(n_feat, feat_dim, n_heads)
        self.from_attn = spectral_norm(nn.Linear(attn_dim, hidden_dim))
        self.out = spectral_norm(nn.Linear(hidden_dim, 1))

    def forward(self, x, class_ids, source_ids):
        e = torch.cat([self.class_emb(class_ids), self.source_emb(source_ids)], dim=-1)
        h = F.leaky_relu(self.proj(torch.cat([x, e], dim=-1)), 0.2)
        for blk in self.blocks: h = h + blk(h)
        a = self.to_attn(h)
        a = self.attn_mod(a)
        h = h + F.leaky_relu(self.from_attn(a), 0.2)
        return self.out(h)

print('Model classes defined.')

Model classes defined.


## 4 · Training

In [10]:
# ── Dataset wrapper ────────────────────────────────────────────
class IDSDataset(Dataset):
    def __init__(self, X, y_class, y_source):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.yc = torch.tensor(y_class,  dtype=torch.long)
        self.ys = torch.tensor(y_source, dtype=torch.long)

    def __len__(self): return len(self.X)
    def __getitem__(self, i): return self.X[i], self.yc[i], self.ys[i]

# ── Inverse-frequency sampler: P(c,s) ∝ 1/sqrt(n_{c,s}) ──────
def make_weighted_sampler(y_class, y_source):
    cs = list(zip(y_class, y_source))
    counts = defaultdict(int)
    for pair in cs: counts[pair] += 1
    weights = np.array([1.0 / (counts[pair]**0.5 + 1e-6) for pair in cs])
    weights /= weights.sum()
    return torch.utils.data.WeightedRandomSampler(
        torch.tensor(weights, dtype=torch.float64),
        num_samples=len(weights), replacement=True)

# ── Gradient penalty ──────────────────────────────────────────
def gradient_penalty(critic, real, fake, class_ids, source_ids, device):
    B = real.shape[0]
    eps = torch.rand(B, 1, device=device)
    hat = (eps * real + (1-eps) * fake).requires_grad_(True)
    score = critic(hat, class_ids, source_ids)
    grad  = torch.autograd.grad(score, hat,
                                grad_outputs=torch.ones_like(score),
                                create_graph=True, retain_graph=True)[0]
    gp = ((grad.norm(2, dim=1) - 1) ** 2).mean()
    return gp

# ── Training function ─────────────────────────────────────────
def train_nsgids(X, y_class, y_source, cfg, variant='full', n_epochs=None):
    """
    variant: 'full' | 'no_attn' | 'no_da' | 'no_gp' | 'no_cond'
    Returns (generator, critic, history)
    """
    use_attn = variant != 'no_attn'
    use_da   = variant != 'no_da'
    use_gp   = variant != 'no_gp'
    use_cond = variant != 'no_cond'
    epochs   = n_epochs or (cfg['FAST_EPOCHS'] if cfg['FAST_EPOCHS'] > 0 else cfg['N_EPOCHS'])

    dataset = IDSDataset(X, y_class, y_source)
    sampler = make_weighted_sampler(y_class, y_source)
    loader  = DataLoader(dataset, batch_size=cfg['BATCH_SIZE'],
                         sampler=sampler, drop_last=True)

    G = Generator(cfg['Z_DIM'], cfg['EMBED_DIM'], cfg['HIDDEN_DIM'],
                  cfg['N_RES_BLOCKS'], prep.output_info,
                  N_CLASSES, N_SOURCES,
                  n_feat=32, feat_dim=cfg['ATTN_FEAT_DIM'],
                  n_heads=cfg['ATTN_HEADS'], dropout=cfg['DROPOUT'],
                  use_attn=use_attn, use_da=use_da).to(DEVICE)

    C = Critic(prep.out_dim, cfg['EMBED_DIM'], cfg['HIDDEN_DIM'],
               cfg['N_RES_BLOCKS'], N_CLASSES, N_SOURCES,
               n_feat=32, feat_dim=cfg['ATTN_FEAT_DIM'],
               n_heads=cfg['ATTN_HEADS'], dropout=cfg['DROPOUT']).to(DEVICE)

    optG = torch.optim.Adam(G.parameters(), lr=cfg['LR'], betas=cfg['BETAS'])
    optC = torch.optim.Adam(C.parameters(), lr=cfg['LR'], betas=cfg['BETAS'])

    history = {'G_loss': [], 'C_loss': [], 'GP': []}
    tau_decay = (cfg['TAU_END'] / cfg['TAU_START']) ** (1.0 / epochs)
    tau = cfg['TAU_START']

    _pct_used = cfg.get('DATA_PCT', 1.0) * 100
    print('[train_nsgids] variant=' + str(variant) +
          f' | epochs={epochs} | data={_pct_used:.0f}% | device={DEVICE}', flush=True)
    epoch_bar = tqdm(range(1, epochs + 1), desc=f'Training ({variant})', leave=True, dynamic_ncols=True)
    for epoch in epoch_bar:
        G.train(); C.train()
        g_losses, c_losses, gps = [], [], []

        batch_bar = tqdm(loader, desc=f'Ep {epoch}/{epochs}', leave=False, total=len(loader), dynamic_ncols=True)
        for real_x, real_c, real_s in batch_bar:
            real_x = real_x.to(DEVICE)
            real_c = real_c.to(DEVICE)
            real_s = real_s.to(DEVICE)
            B = real_x.shape[0]
            if not use_cond:
                real_c = torch.zeros_like(real_c)
                real_s = torch.zeros_like(real_s)

            # ─ critic steps ─
            for _ in range(cfg['CRITIC_STEPS']):
                z = torch.randn(B, cfg['Z_DIM'], device=DEVICE)
                fake_x = G(z, real_c, real_s, tau).detach()
                loss_C = C(fake_x, real_c, real_s).mean() - C(real_x, real_c, real_s).mean()
                if use_gp:
                    gp = gradient_penalty(C, real_x, fake_x, real_c, real_s, DEVICE)
                    loss_C = loss_C + cfg['LAMBDA_GP'] * gp
                    gps.append(gp.item())
                else:
                    # weight clipping (vanilla WGAN)
                    for p in C.parameters(): p.data.clamp_(-0.01, 0.01)
                optC.zero_grad(); loss_C.backward(); optC.step()
                c_losses.append(loss_C.item())

            # ─ generator step ─
            z = torch.randn(B, cfg['Z_DIM'], device=DEVICE)
            fake_x = G(z, real_c, real_s, tau)
            loss_G = -C(fake_x, real_c, real_s).mean()
            optG.zero_grad(); loss_G.backward(); optG.step()
            g_losses.append(loss_G.item())

            batch_bar.set_postfix(
                G=f'{np.mean(g_losses):.4f}',
                C=f'{np.mean(c_losses):.4f}',
                GP=f'{np.mean(gps) if gps else 0.0:.4f}',
                tau=f'{tau:.3f}',
            )

        tau *= tau_decay
        history['G_loss'].append(np.mean(g_losses))
        history['C_loss'].append(np.mean(c_losses))
        history['GP'].append(np.mean(gps) if gps else 0.0)

        epoch_bar.set_postfix(
            G=f'{history["G_loss"][-1]:.4f}',
            C=f'{history["C_loss"][-1]:.4f}',
            GP=f'{history["GP"][-1]:.4f}',
            tau=f'{tau:.3f}',
        )

        if epoch % 50 == 0 or epoch == 1 or epoch == epochs:
            print(f'Epoch {epoch:3d}/{epochs} | G={history["G_loss"][-1]:.4f} '
                  f'C={history["C_loss"][-1]:.4f} τ={tau:.3f}', flush=True)

    return G, C, history

print('Training utilities defined.')

Training utilities defined.


In [11]:
# ══════════════════════════════════════════════════════════════
#  RUN MAIN TRAINING  (NSG-IDS Fused)
#  Set FAST_EPOCHS in CFG for a quick test (e.g. 10 epochs)
# ══════════════════════════════════════════════════════════════
print('Training NSG-IDS (Fused)...')
G_fused, C_fused, hist_fused = train_nsgids(
    X_train, y_train, src_train, CFG, variant='full')

torch.save(G_fused.state_dict(), './models/G_fused.pt')
print('Model saved.')

Training NSG-IDS (Fused)...
[train_nsgids] variant=full | epochs=10 | data=100% | device=mps


Training (full):   0%|          | 0/10 [11:52<?, ?it/s, C=nan, G=nan, GP=nan, tau=0.851]

Epoch   1/10 | G=nan C=nan τ=0.851


Training (full):  90%|█████████ | 9/10 [2:08:38<12:32, 752.06s/it, C=nan, G=nan, GP=nan, tau=0.200]  

Epoch  10/10 | G=nan C=nan τ=0.200


Training (full): 100%|██████████| 10/10 [2:08:38<00:00, 771.90s/it, C=nan, G=nan, GP=nan, tau=0.200]

Model saved.


## 5 · Synthetic Data Generation

In [12]:
@torch.no_grad()
def generate_synthetic(G, n_per_class, label_enc, source_enc,
                        class_to_source=None, batch=2048, tau=0.2):
    """
    Generate n_per_class samples per class.
    class_to_source: dict mapping class_id → preferred source_id
    """
    G.eval()
    if class_to_source is None:
        # assign source by round-robin
        class_to_source = {c: c % N_SOURCES for c in range(N_CLASSES)}

    all_X, all_c = [], []
    for cls_id in range(N_CLASSES):
        src_id = class_to_source[cls_id]
        remaining = n_per_class
        while remaining > 0:
            bs = min(batch, remaining)
            z  = torch.randn(bs, CFG['Z_DIM'], device=DEVICE)
            ci = torch.full((bs,), cls_id, dtype=torch.long, device=DEVICE)
            si = torch.full((bs,), src_id, dtype=torch.long, device=DEVICE)
            x  = G(z, ci, si, tau).cpu().numpy()
            all_X.append(x)
            all_c.extend([cls_id]*bs)
            remaining -= bs

    X_syn = np.concatenate(all_X)
    y_syn = np.array(all_c)
    print(f'Generated {len(X_syn):,} synthetic samples across {N_CLASSES} classes')
    return X_syn, y_syn


# Build class→source mapping from training distribution
from collections import Counter
cs_counts = Counter(zip(train_df['class_id'], train_df['source_id']))
class_to_src = {}
for cid in range(N_CLASSES):
    # pick source with most samples for this class
    best = max(range(N_SOURCES), key=lambda s: cs_counts.get((cid,s), 0))
    class_to_src[cid] = best

X_syn, y_syn = generate_synthetic(
    G_fused, CFG['N_SYNTH_PER_CLASS'], label_enc, source_enc, class_to_src)
np.save('./outputs/X_syn.npy', X_syn)
np.save('./outputs/y_syn.npy', y_syn)
print('Synthetic data saved.')

Generated 260,000 synthetic samples across 13 classes
Synthetic data saved.


## 6 · Evaluation — Statistical Fidelity (Table 3)

In [13]:
# ── decode preprocessed arrays back to raw feature space ──────
def decode_to_raw(X_enc):
    """
    Approximate inverse of CIF32Preprocessor.transform().
    Returns a DataFrame with continuous columns decoded.
    """
    # reconstruct offset for continuous VGM block
    cat_dim = sum(prep.n_cat_classes[c] for c in CAT_COLS)
    vgm_block = X_enc[:, cat_dim: cat_dim + len(CONT_COLS)*(CFG['K_VGM']+1)]
    # rebuild col_slices for VGM
    slices, offset = {}, 0
    for c in CONT_COLS:
        slices[c] = (offset, offset + CFG['K_VGM'] + 1); offset += CFG['K_VGM'] + 1
    prep.vgm.col_slices = slices
    cont_vals = prep.vgm.inverse(vgm_block, CONT_COLS)

    # int bin block
    int_start = cat_dim + len(CONT_COLS)*(CFG['K_VGM']+1)
    int_block = X_enc[:, int_start: int_start + len(INT_COLS)*CFG['N_BINS']]
    islices, ioffset = {}, 0
    for c in INT_COLS:
        islices[c] = (ioffset, ioffset + CFG['N_BINS']); ioffset += CFG['N_BINS']
    prep.bins.col_slices = islices
    int_vals = prep.bins.inverse(int_block, INT_COLS)

    df = pd.DataFrame({**cont_vals, **int_vals})
    return df


# ── KS statistic ──────────────────────────────────────────────
def mean_ks(real_enc, fake_enc):
    """Mean KS statistic across decoded continuous features."""
    real_df = decode_to_raw(real_enc)
    fake_df = decode_to_raw(fake_enc)
    stats = []
    for c in CONT_COLS:
        if c in real_df and c in fake_df:
            r, f = real_df[c].dropna(), fake_df[c].dropna()
            if len(r) > 0 and len(f) > 0:
                ks, _ = ks_2samp(r, f)
                stats.append(ks)
    return float(np.mean(stats)) if stats else float('nan')


# ── Correlation distance ───────────────────────────────────────
def corr_distance(real_enc, fake_enc):
    """Frobenius norm of Pearson correlation matrix difference."""
    def safe_corr(X):
        df = pd.DataFrame(X[:, :min(50, X.shape[1])])  # limit dims for speed
        return df.corr().fillna(0).values
    R = safe_corr(real_enc)
    S = safe_corr(fake_enc)
    return float(np.linalg.norm(R - S, 'fro'))


# ── JSD on categorical columns ─────────────────────────────────
def mean_jsd(real_enc, fake_enc):
    """Mean JSD across one-hot categorical blocks."""
    jsds = []
    offset = 0
    for c in CAT_COLS:
        nc = prep.n_cat_classes[c]
        r  = real_enc[:, offset:offset+nc].mean(0) + 1e-8
        s  = fake_enc[:, offset:offset+nc].mean(0) + 1e-8
        r /= r.sum(); s /= s.sum()
        jsds.append(float(jensenshannon(r, s)))
        offset += nc
    return float(np.mean(jsds)) if jsds else float('nan')


# ── Run fidelity evaluation per source ────────────────────────
def eval_fidelity(G, source_ids_to_eval=None):
    results = {}
    srcs = source_ids_to_eval or test_df['source'].unique()
    for src in srcs:
        src_test = test_df[test_df['source'] == src]
        if len(src_test) < 100: continue
        X_real = prep.transform(src_test)

        # generate synthetic conditioned on same source
        src_id  = int(src)
        classes = src_test['class_id'].unique()
        G_parts = []
        for cid in classes:
            n = min(1000, max(100, (src_test['class_id']==cid).sum()))
            with torch.no_grad():
                z  = torch.randn(n, CFG['Z_DIM'], device=DEVICE)
                ci = torch.full((n,), int(cid), dtype=torch.long, device=DEVICE)
                si = torch.full((n,), src_id,   dtype=torch.long, device=DEVICE)
                xf = G(z, ci, si, 0.2).cpu().numpy()
            G_parts.append(xf)
        X_fake = np.concatenate(G_parts)

        n = min(len(X_real), len(X_fake))
        results[src] = dict(
            KS = mean_ks(X_real[:n], X_fake[:n]),
            CD = corr_distance(X_real[:n], X_fake[:n]),
            JSD= mean_jsd(X_real[:n], X_fake[:n]),
        )
    return results

print('Running fidelity evaluation...')
G_fused.eval()
fidelity = eval_fidelity(G_fused)
print('\n=== Table 3: Statistical Fidelity (NSG-IDS Fused) ===')
print(f'{"Source":15s}  {"KS":>8s}  {"CD":>8s}  {"JSD":>8s}')
src_names = {0:'CIC-IDS-2018', 1:'UNSW-NB15', 2:'NF-ToN-IoT'}
for src, m in sorted(fidelity.items()):
    print(f'{src_names.get(src,str(src)):15s}  {m["KS"]:8.4f}  {m["CD"]:8.4f}  {m["JSD"]:8.4f}')

Running fidelity evaluation...

=== Table 3: Statistical Fidelity (NSG-IDS Fused) ===
Source                 KS        CD       JSD
CIC-IDS-2018          nan   12.6428       nan
UNSW-NB15             nan    9.7876       nan


## 7 · Machine Learning Efficacy — TSTR (Table 4)

In [ ]:
def tstr_eval(X_syn, y_syn, test_sets_by_source, n_estimators=200):
    """
    Train Random Forest on synthetic; evaluate on each real test split.
    Returns dict: source → {F1, Prec, Rec, Acc}
    """
    print(f'  Training RF on {len(X_syn):,} synthetic samples...')
    clf = RandomForestClassifier(n_estimators=n_estimators, n_jobs=-1,
                                  random_state=SEED, class_weight='balanced')
    clf.fit(X_syn, y_syn)

    results = {}
    for src, src_df in test_sets_by_source.items():
        if len(src_df) < 50: continue
        X_r = prep.transform(src_df)
        y_r = src_df['class_id'].values
        yp  = clf.predict(X_r)
        results[src] = dict(
            F1   = f1_score(y_r, yp, average='macro', zero_division=0),
            Prec = precision_score(y_r, yp, average='macro', zero_division=0),
            Rec  = recall_score(y_r, yp, average='macro', zero_division=0),
            Acc  = accuracy_score(y_r, yp),
        )
    return results


print('Running TSTR evaluation (NSG-IDS Fused)...')
mle_fused = tstr_eval(X_syn, y_syn, test_by_src, n_estimators=CFG['RF_TREES'])

# Real-data baseline (train on real, test on real)
print('Running real-data RF baseline...')
mle_real  = tstr_eval(X_train, y_train, test_by_src, n_estimators=CFG['RF_TREES'])

print('\n=== Table 4: TSTR Macro-F1 per Test Set ===')
print(f'{"Method":22s}  {"Source":15s}  {"F1":>6s}  {"Prec":>6s}  {"Rec":>6s}  {"Acc":>6s}')
for src, m in sorted(mle_fused.items()):
    print(f'{"NSG-IDS (Fused)":22s}  {src_names.get(src,str(src)):15s}  '
          f'{m["F1"]:6.4f}  {m["Prec"]:6.4f}  {m["Rec"]:6.4f}  {m["Acc"]:6.4f}')
for src, m in sorted(mle_real.items()):
    print(f'{"Real-Data Baseline":22s}  {src_names.get(src,str(src)):15s}  '
          f'{m["F1"]:6.4f}  {m["Prec"]:6.4f}  {m["Rec"]:6.4f}  {m["Acc"]:6.4f}')

Running TSTR evaluation (NSG-IDS Fused)...
  Training RF on 260,000 synthetic samples...
Running real-data RF baseline...
  Training RF on 540,272 synthetic samples...


## 8 · Attack Coverage (Table 5)

In [ ]:
def coverage_score(y_syn, min_samples=500):
    counts = Counter(y_syn)
    covered = sum(1 for c in range(N_CLASSES) if counts.get(c, 0) >= min_samples)
    return covered, N_CLASSES, 100.0 * covered / N_CLASSES

cov_covered, cov_total, cov_pct = coverage_score(y_syn, CFG['COVERAGE_MIN'])
print(f'\n=== Table 5: Attack Coverage ===')
print(f'NSG-IDS (Fused): {cov_covered}/{cov_total} classes covered ({cov_pct:.1f}%)')
print(f'Min samples threshold: {CFG["COVERAGE_MIN"]}')

# Per-class counts
cnt = Counter(y_syn)
df_cov = pd.DataFrame([
    {'Class': label_enc.classes_[c], 'Count': cnt.get(c,0),
     'Covered': cnt.get(c,0) >= CFG['COVERAGE_MIN']}
    for c in range(N_CLASSES)
]).sort_values('Count', ascending=False)
print(df_cov.to_string(index=False))

## 9 · Ablation Study (Table 6)

In [ ]:
ABLATION_VARIANTS = {
    'w/o Self-Attention': 'no_attn',
    'w/o Discrete-Aware': 'no_da',
    'w/o Gradient Penalty':'no_gp',
    'w/o Class Cond.':    'no_cond',
    'NSG-IDS (Fused)':    'full',
}

ablation_results = {}
# Use a much shorter schedule for ablations/baselines so the notebook stays usable
ABL_EPOCHS = max(3, CFG['FAST_EPOCHS'] if CFG['FAST_EPOCHS'] > 0 else 5)
ABL_CRITIC_STEPS = 1
train_df_abl = train_df  # already subsampled by DATA_PCT
X_train_abl = prep.transform(train_df_abl)
y_train_abl = train_df_abl['class_id'].values
src_train_abl = train_df_abl['source_id'].values

for name, variant in ABLATION_VARIANTS.items():
    if name == 'NSG-IDS (Fused)':
        # reuse already-trained model
        G_abl = G_fused
    else:
        print(f'\nTraining ablation: {name} ({ABL_EPOCHS} epochs)...')
        cfg_abl = dict(CFG)
        cfg_abl['CRITIC_STEPS'] = ABL_CRITIC_STEPS
        cfg_abl['BATCH_SIZE'] = min(CFG['BATCH_SIZE'], 128)
        G_abl, _, _ = train_nsgids(X_train_abl, y_train_abl, src_train_abl, cfg_abl,
                                    variant=variant, n_epochs=ABL_EPOCHS)

    G_abl.eval()
    # generate
    X_a, y_a = generate_synthetic(G_abl, min(1000, CFG['N_SYNTH_PER_CLASS']),
                                   label_enc, source_enc, class_to_src)
    # metrics
    mle_a  = tstr_eval(X_a, y_a, test_by_src, n_estimators=25)
    fid_a  = eval_fidelity(G_abl)
    cov_a  = coverage_score(y_a, CFG['COVERAGE_MIN'])

    mean_f1 = np.mean([m['F1']  for m in mle_a.values()]) if mle_a else 0
    mean_ks_ = np.mean([m['KS'] for m in fid_a.values()]) if fid_a else 0
    mean_cd_ = np.mean([m['CD'] for m in fid_a.values()]) if fid_a else 0

    ablation_results[name] = dict(F1=mean_f1, KS=mean_ks_, CD=mean_cd_, Coverage=cov_a[0])
    print(f'  {name}: F1={mean_f1:.4f} KS={mean_ks_:.4f} CD={mean_cd_:.4f} Cov={cov_a[0]}')

print('\n=== Table 6: Ablation Study ===')
print(f'{"Configuration":25s}  {"F1":>7s}  {"KS":>7s}  {"CD":>7s}  {"Coverage":>8s}')
for name, m in ablation_results.items():
    print(f'{name:25s}  {m["F1"]:7.4f}  {m["KS"]:7.4f}  {m["CD"]:7.4f}  {m["Coverage"]:8d}')

## 10 · Baselines (SMOTE, CTGAN, Vanilla WGAN-GP)

In [ ]:
from imblearn.over_sampling import SMOTE

baseline_results = {}

# ─ SMOTE (single source: CIC-IDS-2018 = source 0) ─
print('Running SMOTE baseline...')
src0_mask  = train_df['source'] == 0
if src0_mask.sum() > 100:
    X_s0  = X_train[src0_mask]
    y_s0  = y_train[src0_mask]
    # only keep classes with >= 6 samples (SMOTE requirement)
    valid  = np.array([c for c in np.unique(y_s0) if (y_s0==c).sum() >= 6])
    mask2  = np.isin(y_s0, valid)
    try:
        sm    = SMOTE(k_neighbors=5, random_state=SEED)
        Xs_sm, ys_sm = sm.fit_resample(X_s0[mask2], y_s0[mask2])
        mle_smote = tstr_eval(Xs_sm, ys_sm, test_by_src, n_estimators=25)
        fid_smote = eval_fidelity(G_fused)  # fidelity vs NSG-IDS real
        cov_smote = coverage_score(ys_sm, CFG['COVERAGE_MIN'])
        baseline_results['SMOTE'] = dict(
            mle=mle_smote, fid=fid_smote, cov=cov_smote[0])
        print(f'  SMOTE: coverage={cov_smote[0]}')
    except Exception as e:
        print(f'  SMOTE failed: {e}')
else:
    print('  Skipping SMOTE: insufficient CIC-IDS-2018 samples')

# ─ Vanilla WGAN-GP (no attn, no discrete-aware) ─
print('Running Vanilla WGAN-GP baseline...')
G_van, _, _ = train_nsgids(X_train_abl, y_train_abl, src_train_abl, cfg_abl,
                            variant='no_attn', n_epochs=ABL_EPOCHS)
G_van.eval()
X_van, y_van = generate_synthetic(G_van, min(1000, CFG['N_SYNTH_PER_CLASS']),
                                   label_enc, source_enc, class_to_src)
mle_van  = tstr_eval(X_van, y_van, test_by_src, n_estimators=25)
fid_van  = eval_fidelity(G_van)
cov_van  = coverage_score(y_van, CFG['COVERAGE_MIN'])
baseline_results['WGAN-GP (Vanilla)'] = dict(mle=mle_van, fid=fid_van, cov=cov_van[0])
print(f'  WGAN-GP Vanilla coverage: {cov_van[0]}')

## 11 · Visualisations

In [ ]:
# ── Training curves ────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
axes[0].plot(hist_fused['G_loss'], label='Generator loss')
axes[0].set_title('Generator Loss'); axes[0].set_xlabel('Epoch'); axes[0].grid(True, alpha=0.3)
axes[1].plot(hist_fused['C_loss'], label='Critic loss', color='orange')
axes[1].set_title('Critic Loss'); axes[1].set_xlabel('Epoch'); axes[1].grid(True, alpha=0.3)
axes[2].plot(hist_fused['GP'], label='Gradient Penalty', color='green')
axes[2].set_title('Gradient Penalty'); axes[2].set_xlabel('Epoch'); axes[2].grid(True, alpha=0.3)
plt.suptitle('NSG-IDS Training Curves', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('./figures/training_curves.pdf', bbox_inches='tight')
plt.show(); print('Saved training_curves.pdf')

In [ ]:
# ── t-SNE: real vs synthetic ───────────────────────────────────
N_TSNE = 2000
rng = np.random.default_rng(SEED)
idx_r = rng.choice(len(X_test),  min(N_TSNE, len(X_test)),  replace=False)
idx_s = rng.choice(len(X_syn),   min(N_TSNE, len(X_syn)),   replace=False)
X_vis = np.concatenate([X_test[idx_r], X_syn[idx_s]])
labels_vis = np.concatenate([np.zeros(len(idx_r)), np.ones(len(idx_s))])
col_med = np.nanmedian(X_vis, axis=0)
nan_mask = np.isnan(X_vis)
X_vis = np.where(nan_mask, np.broadcast_to(col_med, X_vis.shape), X_vis)


print('Running t-SNE (this may take a minute)...')
emb = TSNE(n_components=2, perplexity=40, random_state=SEED, n_iter=500).fit_transform(X_vis)

fig, ax = plt.subplots(figsize=(8,6))
colors = ['#2196F3','#FF5722']
for i, (name, col) in enumerate([('Real', colors[0]), ('Synthetic', colors[1])]):
    mask = labels_vis == i
    ax.scatter(emb[mask,0], emb[mask,1], c=col, s=6, alpha=0.4, label=name)
ax.legend(fontsize=12); ax.set_title('t-SNE: Real vs NSG-IDS Synthetic', fontsize=13, fontweight='bold')
ax.set_xlabel('Dim 1'); ax.set_ylabel('Dim 2')
plt.tight_layout()
plt.savefig('./figures/tsne_real_vs_syn.pdf', bbox_inches='tight')
plt.show(); print('Saved tsne_real_vs_syn.pdf')

In [ ]:
# ── Correlation heatmaps: real vs synthetic ────────────────────
N_FEAT_SHOW = min(15, len(CONT_COLS))  # show first N continuous features

cat_dim = sum(prep.n_cat_classes[c] for c in CAT_COLS)
vgm_len = len(CONT_COLS) * (CFG['K_VGM'] + 1)

def get_cont_block(X_enc):
    block = X_enc[:, cat_dim : cat_dim + vgm_len]
    # extract just the normalized values (every K+1-th element)
    K = CFG['K_VGM']
    norm_vals = np.column_stack([block[:, k*(K+1)+K] for k in range(len(CONT_COLS))])
    return pd.DataFrame(norm_vals[:, :N_FEAT_SHOW],
                        columns=CONT_COLS[:N_FEAT_SHOW])

n_compare = min(3000, len(X_test), len(X_syn))
real_cont = get_cont_block(X_test[:n_compare])
syn_cont  = get_cont_block(X_syn[:n_compare])

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
sns.heatmap(real_cont.corr(), ax=axes[0], cmap='RdBu_r', center=0,
            vmin=-1, vmax=1, square=True, linewidths=0.3,
            xticklabels=True, yticklabels=True, annot=False)
axes[0].set_title('Real Data Correlation', fontsize=12, fontweight='bold')
sns.heatmap(syn_cont.corr(), ax=axes[1], cmap='RdBu_r', center=0,
            vmin=-1, vmax=1, square=True, linewidths=0.3,
            xticklabels=True, yticklabels=True, annot=False)
axes[1].set_title('NSG-IDS Synthetic Correlation', fontsize=12, fontweight='bold')
plt.suptitle('Pearson Correlation Matrices (Continuous CIF-32 Features)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('./figures/correlation_heatmaps.pdf', bbox_inches='tight')
plt.show(); print('Saved correlation_heatmaps.pdf')

In [ ]:
# ── KS bar chart (per-source comparison) ──────────────────────
methods_ks = {}
for src in fidelity.keys():
    methods_ks.setdefault(src_names.get(src,str(src)), {})
    methods_ks[src_names.get(src,str(src))]['NSG-IDS (Fused)'] = fidelity[src]['KS']

if 'WGAN-GP (Vanilla)' in baseline_results:
    for src, m in baseline_results['WGAN-GP (Vanilla)']['fid'].items():
        sn = src_names.get(src,str(src))
        methods_ks.setdefault(sn, {})
        methods_ks[sn]['WGAN-GP (Vanilla)'] = m['KS']

srcs_list = list(methods_ks.keys())
all_methods = list(set(m for d in methods_ks.values() for m in d.keys()))
x = np.arange(len(srcs_list)); width = 0.35

fig, ax = plt.subplots(figsize=(9,5))
palette = ['#1565C0','#E65100','#2E7D32','#6A1B9A']
for i, method in enumerate(all_methods):
    vals = [methods_ks[s].get(method, np.nan) for s in srcs_list]
    ax.bar(x + i*width, vals, width, label=method, color=palette[i % len(palette)], alpha=0.85)
ax.set_xticks(x + width*(len(all_methods)-1)/2)
ax.set_xticklabels(srcs_list, fontsize=11)
ax.set_ylabel('Mean KS Statistic ↓', fontsize=11)
ax.set_title('Statistical Fidelity (KS) per Source Dataset', fontsize=13, fontweight='bold')
ax.legend(fontsize=10); ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('./figures/ks_bar_chart.pdf', bbox_inches='tight')
plt.show(); print('Saved ks_bar_chart.pdf')

In [ ]:
# ── Attack coverage chart ──────────────────────────────────────
method_cov = {'NSG-IDS\n(Fused)': cov_covered}
if 'WGAN-GP (Vanilla)' in baseline_results:
    method_cov['WGAN-GP\n(Vanilla)'] = baseline_results['WGAN-GP (Vanilla)']['cov']
if 'SMOTE' in baseline_results:
    method_cov['SMOTE'] = baseline_results['SMOTE']['cov']

fig, ax = plt.subplots(figsize=(8,5))
bars = ax.bar(method_cov.keys(), method_cov.values(),
              color=['#1565C0','#E65100','#2E7D32','#6A1B9A'][:len(method_cov)], alpha=0.85)
ax.axhline(N_CLASSES, color='red', linestyle='--', linewidth=1.5, label=f'Total ({N_CLASSES} classes)')
ax.bar_label(bars, fmt='%d', padding=2, fontsize=11)
ax.set_ylabel('Attack Classes Covered', fontsize=11)
ax.set_ylim(0, N_CLASSES + 3)
ax.set_title('Attack Class Coverage (≥500 synthetic samples)', fontsize=13, fontweight='bold')
ax.legend(fontsize=10); ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('./figures/coverage_chart.pdf', bbox_inches='tight')
plt.show(); print('Saved coverage_chart.pdf')

In [ ]:
# ── Synthetic distribution per class ──────────────────────────
cnt_series = pd.Series({label_enc.classes_[c]: cnt.get(c,0) for c in range(N_CLASSES)})
cnt_series = cnt_series.sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(8, max(5, N_CLASSES*0.35)))
colors_bar = ['#C62828' if v < CFG['COVERAGE_MIN'] else '#1565C0' for v in cnt_series]
cnt_series.plot(kind='barh', ax=ax, color=colors_bar, alpha=0.85)
ax.axvline(CFG['COVERAGE_MIN'], color='red', linestyle='--', linewidth=1.5,
           label=f'Threshold ({CFG["COVERAGE_MIN"]})')
ax.set_xlabel('Synthetic Samples Generated', fontsize=11)
ax.set_title('Synthetic Samples per Attack Class (NSG-IDS Fused)',
             fontsize=13, fontweight='bold')
ax.legend(fontsize=10); ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.savefig('./figures/class_distribution.pdf', bbox_inches='tight')
plt.show(); print('Saved class_distribution.pdf')

In [ ]:
# ── Feature KS heatmap (per class, per feature) ───────────────
N_FEA_SHOW = min(10, len(CONT_COLS))
N_CLS_SHOW = min(10, N_CLASSES)
ks_matrix  = np.zeros((N_CLS_SHOW, N_FEA_SHOW))

with torch.no_grad():
    for ci in range(N_CLS_SHOW):
        real_ci = X_test[y_test == ci][:500]
        if len(real_ci) < 20: continue
        z  = torch.randn(len(real_ci), CFG['Z_DIM'], device=DEVICE)
        c_ = torch.full((len(real_ci),), ci, dtype=torch.long, device=DEVICE)
        s_ = torch.full((len(real_ci),), class_to_src.get(ci,0), dtype=torch.long, device=DEVICE)
        fake_ci = G_fused(z, c_, s_, 0.2).cpu().numpy()
        for fi, feat in enumerate(CONT_COLS[:N_FEA_SHOW]):
            real_f = decode_to_raw(real_ci)
            fake_f = decode_to_raw(fake_ci)
            if feat in real_f.columns and feat in fake_f.columns:
                r_f, f_f = real_f[feat].dropna(), fake_f[feat].dropna()
                if len(r_f) > 0 and len(f_f) > 0:
                    ks, _ = ks_2samp(r_f, f_f)
                    ks_matrix[ci, fi] = ks

fig, ax = plt.subplots(figsize=(12, 6))
sns.heatmap(ks_matrix,
            xticklabels=[c.replace('_',' ') for c in CONT_COLS[:N_FEA_SHOW]],
            yticklabels=[label_enc.classes_[i] for i in range(N_CLS_SHOW)],
            cmap='YlOrRd', vmin=0, vmax=0.5, ax=ax,
            annot=True, fmt='.2f', linewidths=0.3)
ax.set_title('Per-Class KS Statistic (lower = better fidelity)',
             fontsize=13, fontweight='bold')
ax.set_xlabel('CIF-32 Continuous Feature', fontsize=11)
ax.set_ylabel('Attack Class', fontsize=11)
plt.tight_layout()
plt.savefig('./figures/ks_heatmap.pdf', bbox_inches='tight')
plt.show(); print('Saved ks_heatmap.pdf')

## 12 · Final Results Tables (Paper-Ready)

In [ ]:
print('=' * 65)
print('  FINAL RESULTS — copy these into NSG-IDS.tex (replace \\nv{})')
print('=' * 65)

# Table 3
print('\n--- Table 3: Statistical Fidelity ---')
print(f'{"Method":<22} {"Source":<15} {"KS":>8} {"CD":>8} {"JSD":>8}')
for src, m in sorted(fidelity.items()):
    sn = src_names.get(src, str(src))
    print(f'{"NSG-IDS (Fused)":<22} {sn:<15} {m["KS"]:8.4f} {m["CD"]:8.4f} {m["JSD"]:8.4f}')

# Table 4
print('\n--- Table 4: TSTR Macro-F1 ---')
print(f'{"Method":<22} {"Source":<15} {"F1":>7} {"Prec":>7} {"Rec":>7} {"Acc":>7}')
for src, m in sorted(mle_fused.items()):
    sn = src_names.get(src, str(src))
    print(f'{"NSG-IDS (Fused)":<22} {sn:<15} {m["F1"]:7.4f} {m["Prec"]:7.4f} {m["Rec"]:7.4f} {m["Acc"]:7.4f}')
for src, m in sorted(mle_real.items()):
    sn = src_names.get(src, str(src))
    print(f'{"Real-Data Baseline":<22} {sn:<15} {m["F1"]:7.4f} {m["Prec"]:7.4f} {m["Rec"]:7.4f} {m["Acc"]:7.4f}')

# Table 5
print(f'\n--- Table 5: Coverage ---')
print(f'NSG-IDS (Fused): {cov_covered} / {N_CLASSES}  ({cov_pct:.1f}%)')
if 'WGAN-GP (Vanilla)' in baseline_results:
    v = baseline_results['WGAN-GP (Vanilla)']['cov']
    print(f'WGAN-GP Vanilla: {v} / {N_CLASSES}  ({100*v/N_CLASSES:.1f}%)')

# Table 6
print('\n--- Table 6: Ablation ---')
print(f'{"Configuration":<25} {"F1":>7} {"KS":>7} {"CD":>7} {"Coverage":>8}')
for name, m in ablation_results.items():
    print(f'{name:<25} {m["F1"]:7.4f} {m["KS"]:7.4f} {m["CD"]:7.4f} {m["Coverage"]:8d}')

# Model size
n_params = sum(p.numel() for p in G_fused.parameters())
print(f'\n--- Implementation Details ---')
print(f'Generator parameters: {n_params/1e6:.2f}M')
print(f'Synthetic samples generated: {len(X_syn):,} ({N_CLASSES} classes × {CFG["N_SYNTH_PER_CLASS"]:,})')

# Save results JSON
results_out = dict(
    fidelity={
        src_names.get(s,str(s)): {k: round(v,5) for k,v in m.items()}
        for s,m in fidelity.items()
    },
    tstr={
        src_names.get(s,str(s)): {k: round(v,5) for k,v in m.items()}
        for s,m in mle_fused.items()
    },
    coverage=dict(covered=cov_covered, total=N_CLASSES, pct=round(cov_pct,2)),
    ablation={name: {k: round(v,5) if isinstance(v,float) else v for k,v in m.items()}
              for name,m in ablation_results.items()},
    model_params=n_params,
)
with open('./outputs/results.json', 'w') as f:
    json.dump(results_out, f, indent=2)
print('\nAll results saved to ./outputs/results.json')
print('All figures saved to ./figures/')